In [ ]:
import colorcet as cc
import gstools as gs
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from context_flux_no.simulations.pde.cubic import CubicFlux2D
from context_flux_no.simulations.utils import generate_dataset
from context_flux_no.waveforms.grf import GaussianRandomField2D

In [ ]:
x, y = jnp.linspace(0, 1, 64), jnp.linspace(0, 1, 64)
grf = GaussianRandomField2D(gs.Gaussian(dim=2, var=1, len_scale=0.3))
u0 = grf.sample((x, y), jax.random.key(0))

In [ ]:
u0.shape

In [ ]:
cubic_flux = CubicFlux2D(a=-1.0, b=1.0, c=1.0)
sol = cubic_flux.solve(
    lambda xs: grf.sample(xs, jax.random.key(1)),
    ((0, 1), (0, 1)),
    (50, 50),
    (0, 0.5),
    100,
)

In [ ]:
sol

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(sol["values"][0, i * 10, 0], cmap=cc.cm.coolwarm)

In [ ]:
dataset = generate_dataset(
    n_coeffs=10,
    n_ics_per_coeff=10,
    pde_factory=CubicFlux2D,
    initial_condition_fn=GaussianRandomField2D(
        gs.Gaussian(dim=2, var=1, len_scale=0.3)
    ).sample,
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_spans=((0, 1), (0, 1)),
    Nxs=(50, 50),
    t_span=(0, 0.5),
    Nt=100,
    seed=0,
)

In [ ]:
dataset["values"].nbytes / 1e9